<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/11_gemini_grounding_caching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Needs a Google API key.** Search grounding and context caching are native Gemini features, so this notebook calls `google.genai` directly rather than routing through OpenRouter. It ships without stored outputs because it cannot run on the OpenRouter-only setup used for the rest of the course. To run it, open it in Colab (badge above), set a `GOOGLE_API_KEY` (from Google AI Studio) in the environment or Colab secrets, and run the cells.

# Module 11 — Gemini Unlocks: Search Grounding and Context Caching

> **⚡ Quick path** — this is one of the four modules of the 1-hour course preview (M01 → M02 → M05 → M11).


> **Where you are** — Part 2 begins: what you *gain* if you commit to Gemini. (MVP viewers: only this module is filmed; M12–M14 are self-study notebooks in the same style.)
> - **You can already:** hosted tools from the previous course — you used OpenAI's `web_search` there; grounding is Gemini's built-in version of that idea.
> - **New in this module:** the native `google-genai` client and context caching.

**Part 2 of the course begins here.**

Ten modules of vendor-neutral ADK — everything running on whichever model you chose. Part 2 is honest about the other side of that deal: what you give up by staying vendor-neutral. This module covers two capabilities that nothing in the LiteLLM world can replicate:

- **Google Search grounding** — Gemini calls Google Search as a built-in tool and answers with citations to *real URLs*, plus metadata showing which source backs each claim. No search-API wrangling, no hallucinated links. (Billed separately: ~$35 per 1,000 grounded requests.)
- **Long context + context caching** — Gemini 2.5+ takes 1M-token inputs, and for repeat questions over the same big document you can **cache** it once and get a 75–90% discount afterwards. This is what makes long-document agents affordable.

### The switch is one line

Remember the side-by-side from the very first module — the greeter with `model="gemini-2.5-flash"` as a plain string versus the `LiteLlm(...)` wrapper? Part 2 simply uses the plain-string form. Everything else about an agent — tools, sessions, workflow agents, callbacks — stays exactly as you learned it.

And where we talk to Gemini *outside* ADK in this module, `genai.Client(api_key=...)` is simply Google's version of the `OpenAI(...)` client you used all through the previous course. Same idea, different vendor.

### Why gemini-2.5-flash and not the newest one?

Google ships Flash models fast (3.1 → 3.5 → 3.6 → 3.7, the last released 2026-08-13). We stay on the 2.5 line because it is still current, not scheduled for shutdown, the cheapest, and its grounding/caching behaviour is stable — the newest models were returning *503 high demand* errors when this notebook was last refreshed (2026-08-23). Moving later is one string: `model="gemini-3.7-flash"`. Check [the pricing page](https://ai.google.dev/gemini-api/docs/pricing) before switching.

# Setup

In [1]:
!pip install -q google-adk==2.7.1 google-genai litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

✅ Packages installed.


## API Key — Switching to Google AI Studio

Part 1 used `OPENROUTER_API_KEY`. Part 2 uses `GOOGLE_API_KEY` from [aistudio.google.com/apikey](https://aistudio.google.com/apikey). The free tier is enough for M11–M13; a paid tier unlocks context caching and higher rate limits.

`GOOGLE_GENAI_USE_VERTEXAI=FALSE` (the default) keeps you on Google AI Studio; `TRUE` switches to Vertex AI, which needs GCP billing.

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")

GOOGLE_API_KEY = None
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
        if GOOGLE_API_KEY: print("✅ API key loaded from .env file.")
    except ImportError: pass
if not GOOGLE_API_KEY:
    from getpass import getpass
    print("💡 Get one at aistudio.google.com/apikey (free tier works).")
    GOOGLE_API_KEY = getpass("Enter your Google AI Studio API key: ")
assert GOOGLE_API_KEY, "❌ No API key."
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
print("✅ Environment ready for Gemini.")

✅ API key loaded from .env file.
✅ Environment ready for Gemini.


## Imports

Two imports to notice, both new: **`google_search`** from `google.adk.tools` — the built-in tool this module is about — and **`from google import genai`**, the direct Google SDK we'll use for caching. The rest is the same as Part 1.

In [3]:
import asyncio, uuid, logging
import nest_asyncio; nest_asyncio.apply()
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

# ADK — same as Part 1
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search   # ← new: the Gemini-only built-in
from google.genai import types

# Direct google-genai SDK for caching and raw long-context calls
from google import genai

print("✅ Imports successful.")

✅ Imports successful.


# Google Search Grounding

Every agent you built in Part 1 had a knowledge cutoff. Ask about today's news and you got a refusal — or a confident wrong answer. In the previous course you saw one fix: OpenAI's hosted `web_search` tool. Gemini's version of that idea is called **grounding**, and in ADK it is one import: add `google_search` to the agent's `tools=` list and every query that needs fresh information triggers a real Google Search, with the sources reported back to you.

**Cost note:** ~$35 per 1,000 grounded requests, billed separately from tokens. Budget accordingly.

### The key lines, before you run them

```python
model="gemini-2.5-flash",   # plain string — the native path, no adapter
tools=[google_search],       # imported, not written
```

Both lines say the same calming thing: nothing new to build. The model slot takes the plain string (native Gemini needs no `LiteLlm` wrapper), and the tool is one you *import* instead of write — the search itself runs on Google's side.

In [4]:
# Notice: plain string model name, not LiteLlm(). This is the native path.
grounded_agent = LlmAgent(
    name="grounded_agent",
    model="gemini-2.5-flash",
    description="Answers questions with up-to-date Google Search results.",
    instruction=(
        "You answer user questions using Google Search for anything that "
        "requires current information. Cite your sources by URL in your reply."
    ),
    tools=[google_search],
)

print("✅ grounded_agent ready.")

✅ grounded_agent ready.


Same `chat()` helper as the whole course, with one addition: when an event carries **`grounding_metadata`** — the list of real sources Gemini used — the helper prints it.

In [5]:
APP = "m11"
USER = "student"
session_service = InMemorySessionService()

async def chat(agent, prompt: str):
    sid = f"s-{uuid.uuid4().hex[:6]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER: {prompt}\n")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text and p.text.strip():
                    print(f"[{ev.author}] {p.text.strip()[:400]}")
                if p.function_call:
                    print(f"[tool_call] {p.function_call.name}({dict(p.function_call.args or {})})")
        if ev.grounding_metadata:
            chunks = getattr(ev.grounding_metadata, "grounding_chunks", None) or []
            if chunks:
                print(f"\n── Grounding metadata: {len(chunks)} source(s) ──")
                for i, chunk in enumerate(chunks[:3], 1):
                    web = getattr(chunk, "web", None)
                    if web:
                        print(f"  {i}. {getattr(web, 'title', '?')}")
                        print(f"     {getattr(web, 'uri', '?')[:100]}")

await chat(grounded_agent, "What is the capital of Slovakia and its current population?")

USER: What is the capital of Slovakia and its current population?



Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


[grounded_agent] The capital of Slovakia is Bratislava.

As of August 17, 2026, the current population of Slovakia is estimated to be 5,438,788 people. Another source, Countrymeters, estimates the population to be 5,484,005 as of August 12, 2026.

── Grounding metadata: 6 source(s) ──
  1. ricksteves.com
     https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQETHFwXEgrwTg9vgtIwiYFEVDRFWqv77
  2. wikipedia.org
     https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHT5toEnfkhOiMxg7q5Bs1X6CfyXfXEu
  3. mapy.com
     https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEr3FOyvDGlSchk_VdNF-BZ4MzK9oCR8


### 🔍 What just happened?

- **No `[tool_call]` line appeared.** That's not a bug. `google_search` is a *built-in* tool: Gemini runs the search inside its own infrastructure, so ADK never runs a Python function on your side. You see the result of grounding, not the call itself — same as the hosted `web_search` tool in the previous course.
- **The `grounding_metadata` block lists real URLs.** These are the actual pages Gemini used, not invented links. In a production agent you would show them to users as citations under each answer.

This is the biggest difference you'll feel between Part 1 and Part 2: in Part 1 you would write `async def web_search(query): ...`, pay for a search API, parse JSON, feed it back. Here you added one import.

### 🎯 Mini-task

Change `grounded_agent`'s instruction to require **at least two sources** from the grounding metadata in every answer. Ask it a question. Does it comply?

# One Rule to Remember — Built-in Tools Don't Combine

One sharp edge before you build on this. **Gemini's built-in tools (`google_search`, code execution, Vertex AI Search) cannot sit next to other tools in the same agent** — the API rejects the request:

```python
# This fails: google_search is a built-in; get_weather is a regular function tool
BAD = LlmAgent(
    model="gemini-2.5-flash",
    tools=[google_search, get_weather],   # ← mixing built-in + regular
)
```

The usual fix is the wrapping move you already know from the translator specialist: give each built-in tool its own small agent and wrap it with `AgentTool`. The coordinator calls the search specialist for search questions and the weather agent for weather — each child keeps a clean tool list.

(For Search specifically, ADK ≥ 1.16 also accepts `bypass_multi_tools_limit=True`. Check your version before relying on it.)

### 🎯 Mini-task

Try it: add a small `get_weather` function tool next to `google_search` and read the error. Then refactor with `AgentTool` wrappers so both work.

# Self-Study from Here — Long Context and Context Caching

The filmed part of this module ends here. What follows is self-study in the same style — and it's worth your time, because this is the money-saving half.

The setup: Gemini 2.5+ accepts up to **1M input tokens** (2M on Pro). You can paste a 500-page manual, a whole codebase, or months of email into one request. The catch is cost: ask ten questions against the same 500K-token manual and you've paid for 5M input tokens — roughly $10 on Flash, $50 on Pro.

**Context caching** fixes exactly this: store the big document once, and every later question against it is billed at a 75–90% discount on the cached part. Two flavors:

- **Implicit caching** — Gemini 2.5+ automatically caches repeated prefixes. Zero code changes, ~75% discount.
- **Explicit caching** — `client.caches.create(...)` with a time-to-live you choose. Your control, ~90% discount — but it needs a **paid tier** (the free tier's caching quota is zero).

We'll run a long-context query first (no caching), then look at the explicit-caching pattern in code.

## A Long-Context Query

The next cell steps outside ADK for a moment. The key line:

```python
client = genai.Client(api_key=GOOGLE_API_KEY)
```

This is Google's version of the `OpenAI(...)` client from the previous course — you create it once and call `client.models.generate_content(...)` the way you called `client.responses.create(...)`. We use it directly here so we can read the raw token counts, which is the whole story in this section.

In [6]:
client = genai.Client(api_key=GOOGLE_API_KEY)

# A small manual. In production this could be 100KB+.
MANUAL = '''\
RaspiKitchen v2.3 — Technical Manual.

CHAPTER 1 — Overview.
The device is a voice-controlled kitchen assistant running on a Raspberry
Pi 5 with 8GB RAM and an ESP32-S3 microphone array.

CHAPTER 2 — Installation.
Power: USB-C PD, 45W minimum. Network: WiFi 6 or Ethernet. Mount the
microphone array at eye level, 1-2m from the workspace.

CHAPTER 3 — Voice wake word.
Default wake word is "Chef". To change, edit /etc/raspikitchen/wake.conf
and restart the raspikitchen.service.

CHAPTER 4 — Tool list.
Supported tools: timer, recipe lookup, conversion, substitution, shopping
list, thermometer polling (via BLE).

CHAPTER 5 — Privacy.
Audio is processed locally by default. Cloud fallback can be enabled via
/etc/raspikitchen/cloud.conf but is OFF by default.

CHAPTER 6 — Troubleshooting.
If the device does not respond to wake word, check microphone array
connection on GPIO pins 22-27. If voice recognition is slow, verify the
local Whisper model is loaded (ps aux | grep whisper).

CHAPTER 7 — Updating.
Run: sudo raspikitchen-update. Will pull the latest image and restart on
completion. Takes 2-5 minutes.
'''

# A single direct call — no ADK, just google-genai — so we see raw token counts.
resp = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"Based on this manual, what is the default wake word and how do I change it?\n\nMANUAL:\n{MANUAL}",
)
print(f"Response: {resp.text[:500]}")
print(f"\nToken usage:")
print(f"  input:  {resp.usage_metadata.prompt_token_count}")
print(f"  output: {resp.usage_metadata.candidates_token_count}")
print(f"  total:  {resp.usage_metadata.total_token_count}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Response: Based on the manual:

*   The **default wake word** is "Chef".
*   To **change it**, you need to edit the file `/etc/raspikitchen/wake.conf` and then restart the `raspikitchen.service`.

Token usage:
  input:  323
  output: 54
  total:  765


### 🔍 What just happened?

- **`usage_metadata.prompt_token_count`** is what you paid on the input side. For a real 100KB manual that could be 25,000+ tokens *per question* — ten questions means 250,000 input tokens.
- **No caching happened yet.** Run the same query twice and you're billed for the manual twice. That's the problem caching solves.

### 🎯 Mini-task

Make `MANUAL` 50× longer (`MANUAL * 50`). What's the input token count now? Roughly how many copies would it take to hit the 1M-token limit?

## Explicit Context Caching

The pattern, decoded before you run it:

```python
cache = client.caches.create(model=..., config=...)   # pin the big document to Gemini's wall
resp  = client.models.generate_content(..., config=GenerateContentConfig(cached_content=cache.name))
```

You create the cache once — contents, an optional system instruction, and a `ttl` (how long it lives; storage is billed by the second). Then every call that passes `cached_content=cache.name` reuses the pinned document at ~10% of the normal input price.

⚠️ **This cell will most likely fail on the free tier** — explicit caching has a zero quota there. The code is still the correct pattern; on a paid key it completes. Read it either way.

In [7]:
# Make the cache payload larger — caches have a ~32K-token minimum.
big_manual = MANUAL * 50   # repeat the manual 50 times to hit the cache threshold
approx_tokens = len(big_manual.split()) * 1.3
print(f"Cache payload: ~{int(approx_tokens)} tokens (minimum is ~32,768)")
print()

try:
    cache = client.caches.create(
        model="gemini-2.5-flash",
        config=types.CreateCachedContentConfig(
            contents=[types.Content(role="user", parts=[types.Part(text=big_manual)])],
            system_instruction="Answer questions about the RaspiKitchen manual.",
            ttl="300s",   # 5 minutes; charged by the second
        ),
    )
    print(f"✅ Cache created: {cache.name}")
    print(f"   cached_content_token_count: {cache.usage_metadata.total_token_count}")

    # Now query against the cache. The 32K+ cached tokens are billed at ~10% of normal rate.
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="What is the default wake word?",
        config=types.GenerateContentConfig(cached_content=cache.name),
    )
    print(f"\nResponse: {resp.text[:200]}")
    print(f"Tokens — cached: {resp.usage_metadata.cached_content_token_count}, "
          f"prompt: {resp.usage_metadata.prompt_token_count}, "
          f"output: {resp.usage_metadata.candidates_token_count}")

    # Clean up — caches have a TTL but you can delete explicitly
    client.caches.delete(name=cache.name)
    print(f"✅ Cache deleted.")

except Exception as e:
    print(f"❌ Caching unavailable on this tier.")
    print(f"   Error: {str(e)[:200]}")
    print()
    print("   The free tier has TotalCachedContentStorageTokensPerModelFreeTier = 0.")
    print("   Context caching requires a paid Gemini API tier.")
    print("   The code above IS the correct pattern; it fails at the billing gate,")
    print("   not at the API surface. On a paid key, this cell completes.")

Cache payload: ~11115 tokens (minimum is ~32,768)



❌ Caching unavailable on this tier.
   Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'TotalCachedContentStorageTokensPerModelFreeTier limit exceeded for model gemini-2.5-flash: limit=0, requested=14961', 'status': 'RESOURCE_EX

   The free tier has TotalCachedContentStorageTokensPerModelFreeTier = 0.
   Context caching requires a paid Gemini API tier.
   The code above IS the correct pattern; it fails at the billing gate,
   not at the API surface. On a paid key, this cell completes.


## What Does Caching Save? The Numbers

Gemini 2.5 Flash pricing (April 2026 — verify current rates):

| Token type | Rate |
|---|---|
| Standard input | $0.075 per 1M tokens |
| Cached input (explicit) | ~$0.008 per 1M tokens (~90% discount) |
| Cache storage | ~$0.01 per 1M tokens per hour |
| Output | $0.30 per 1M tokens |

**Worked example.** A 100-page PDF ≈ 50K tokens; your agent gets 20 questions per hour about it.

- **No caching:** 20 × 50K input = 1M tokens/hour → **$0.075/hour**.
- **Explicit caching:** one full-price insertion + 20 cheap cached reads + storage = **~$0.013/hour**.

Roughly **6× cheaper**, and the gap widens with more questions. The rule of thumb: below ~3–5 questions per document per hour, free implicit caching already gives you most of the benefit; above that, explicit caching earns its extra code.

### 🎯 Mini-task

Run the long-context query twice in a row *without* explicit caching. Does the second response show a non-zero `cached_content_token_count`? (That's implicit caching at work.)

# Gemini-Native or LiteLLM? How to Choose

You can use Gemini through LiteLLM (Part 1 style) or natively (this module's style). Here is the honest trade-off:

| Feature | Native (`model="gemini-..."`) | LiteLLM (`LiteLlm(model="openrouter/google/...")`) |
|---|---|---|
| Basic chat / tool calls | ✅ | ✅ |
| `google_search` built-in tool | ✅ | ❌ |
| Built-in code execution | ✅ | ❌ |
| Context caching | ✅ | ❌ |
| Thinking budgets | ✅ | ⚠️ partial via `reasoning` param |
| Live API / voice | ✅ | ❌ |
| Switch to Claude/GPT/Qwen in one line | ❌ | ✅ |

**The practical rule:** go native when you need a feature that doesn't travel through LiteLLM; stay wrapped for everything else, so your code keeps its portability. Production systems often run *both* — a portable LiteLLM agent for routine work and a native Gemini agent for the tasks that need grounding, long context, or voice.

# Key Takeaways

- **`google_search`** is a Gemini-only built-in tool: add it to `tools=`, Gemini searches internally, and `grounding_metadata` gives you real citation URLs. ~$35 per 1,000 grounded requests.
- **Built-in tools don't combine** with regular tools in one agent — wrap each in its own small agent with `AgentTool` (or use `bypass_multi_tools_limit=True` for Search on ADK ≥ 1.16).
- **Long context**: 1M input tokens on Gemini 2.5+ — room for whole documents.
- **Implicit caching** is automatic (~75% off repeated prefixes); **explicit caching** (`client.caches.create`) reaches ~90% off but needs a paid tier.
- **Choose native Gemini** for the features LiteLLM can't reach; stay wrapped for portability; production systems often run both.

# Next up — M12: Thinking Budgets

Gemini 2.5+ has a knob that trades speed for reasoning depth: `ThinkingConfig`. Set the budget to zero for instant answers, or to 8,000+ tokens for hard problems. We'll run the same problem at both extremes and watch the answer quality — and the bill — change.